# Transactions and ACID Principles

Imagine you are transferring $100 from your bank account to your friend's account. This requires two steps:
1. Deduct $100 from your account.
2. Add $100 to your friend's account.

What happens if the database crashes exactly after step 1, but before step 2? Your money disappears into the void! 

To prevent this, databases use **Transactions**. A transaction treats multiple SQL statements as a single, indivisible unit of work. They either *all* succeed together, or they *all* fail together.

Let's use Python and SQLite to simulate a bank transfer and see how transactions protect our data.

In [1]:
import sqlite3
import pandas as pd

# 1. Connect to an in-memory database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 2. Create an Accounts table with a strict rule: Balances cannot go below zero!
cursor.executescript("""
CREATE TABLE Accounts (
    account_id INTEGER PRIMARY KEY,
    name TEXT,
    balance DECIMAL(10, 2) CHECK(balance >= 0) -- This is a Consistency constraint!
);

INSERT INTO Accounts (name, balance) VALUES 
('Alice', 1000.00),
('Bob', 500.00);
""")

print("✅ Bank Accounts created!")
display(pd.read_sql_query("SELECT * FROM Accounts;", conn))

✅ Bank Accounts created!


,account_id,name,balance
0,1,Alice,1000
1,2,Bob,500


# 1. The ACID Principles
Before we write the code, we need to understand the four pillars of a reliable database, known as **ACID**:

* **A - Atomicity (All or Nothing)**: A transaction must fully complete. If any part of it fails, the entire transaction is rolled back as if nothing ever happened.
* **C - Consistency (Follow the Rules)**: A transaction can only bring the database from one valid state to another valid state. It cannot break database rules (like our `CHECK(balance >= 0)` rule).
* **I - Isolation (No Interference)**: If 1,000 people are making bank transfers at the exact same time, the database processes them in a way that ensures their transactions do not overlap or corrupt each other. 
* **D - Durability (Permanent)**: Once a transaction is successfully saved (committed), it is permanent. Even if the server loses power a second later, the data is safe on the hard drive.

# 2. A Successful Transaction (`COMMIT`)
In Python's `sqlite3` library, a transaction starts automatically when you execute a command. To finalize the changes and make them permanent, you must explicitly call `conn.commit()`.

In [2]:
# Let's transfer $200 from Alice to Bob
try:
    # Step 1: Deduct from Alice
    cursor.execute("UPDATE Accounts SET balance = balance - 200 WHERE name = 'Alice';")
    
    # Step 2: Add to Bob
    cursor.execute("UPDATE Accounts SET balance = balance + 200 WHERE name = 'Bob';")
    
    # Step 3: If both steps succeed, permanently save the changes!
    conn.commit()
    print("✅ Transfer successful! Changes committed.")

except Exception as e:
    # If anything goes wrong, undo everything.
    conn.rollback()
    print("❌ Transfer failed!")

display(pd.read_sql_query("SELECT * FROM Accounts;", conn))

✅ Transfer successful! Changes committed.


,account_id,name,balance
0,1,Alice,800
1,2,Bob,700


# 3. A Failed Transaction (`ROLLBACK`)
Now let's see what happens when things go wrong. We will try to make Alice transfer $5,000 to Bob. 

Remember, Alice only has $800 left, and we added a `CHECK(balance >= 0)` rule when we built the table. Let's see how the transaction handles this error.

In [3]:
print("Attempting to transfer $5000 from Alice to Bob...")

try:
    # Step 1: Attempt to deduct from Alice (This will cause an error!)
    cursor.execute("UPDATE Accounts SET balance = balance - 5000 WHERE name = 'Alice';")
    
    # Step 2: Add to Bob (The code will never reach this line)
    cursor.execute("UPDATE Accounts SET balance = balance + 5000 WHERE name = 'Bob';")
    
    # Step 3: Save changes
    conn.commit()

except sqlite3.IntegrityError as e:
    # ATOMICITY IN ACTION: We catch the error and undo the entire transaction
    conn.rollback()
    print(f"❌ Transfer failed and ROLLED BACK due to error: {e}")

# Let's check the balances. Because of the rollback, Bob's money wasn't touched, 
# and the database remains perfectly consistent!
print("\n--- Final Balances (Data remains safe) ---")
display(pd.read_sql_query("SELECT * FROM Accounts;", conn))

# Clean up
conn.close()

Attempting to transfer $5000 from Alice to Bob...
❌ Transfer failed and ROLLED BACK due to error: CHECK constraint failed: balance >= 0

--- Final Balances (Data remains safe) ---


,account_id,name,balance
0,1,Alice,800
1,2,Bob,700


# 4. Transactions in Data Science
As a Data Scientist, you won't build banking apps, but you will build Data Pipelines (ETL). 

If you are inserting 10 million cleaned rows into a central database, and row 9,999,999 has a bad data type that crashes the script, you do not want 9,999,998 rows lingering in the database. Wrapping your pipeline in a `TRY / EXCEPT / ROLLBACK` block ensures that either all 10 million rows are inserted perfectly, or zero rows are inserted, keeping your analytical data pure.

---

## Real-World Use Case or Analogy:
Think of Transactions and ACID like buying a snack from a **Vending Machine**:

* **Atomicity**: You put a dollar in, press the button, and the coil spins. Either the snack falls into the tray AND it keeps your dollar (Success), OR the snack gets stuck, and the machine spits your dollar back out (Rollback). It never keeps your dollar while leaving you hungry.
* **Consistency**: The machine only accepts valid currency. If you try to put a shiny button or a piece of plastic into the coin slot, it rejects it immediately to protect the machine's internal mechanics.
* **Isolation**: Imagine a giant vending machine with two coin slots. You and a stranger insert money and push buttons at the exact same millisecond. The machine's computer briefly pauses one of you to ensure the robotic arms don't crash into each other trying to grab the same bag of chips.
* **Durability**: Once the snack drops and you have it in your hands, the transaction is over. Even if someone unplugs the vending machine from the wall right after, your snack cannot be magically taken back.

---